In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": [
    "# Phase 2: Model Training\n",
    "**Project:** Osteo-Analyst\n",
    "\n",
    "**Objective:** Train the DenseNet121 model to classify Knee X-rays into 5 KL grades. This notebook implements Transfer Learning using the custom architecture defined in `backend/model.py`."
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "import sys\n",
    "import os\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "from torch.utils.data import DataLoader\n",
    "from torchvision import datasets, transforms\n",
    "from pathlib import Path\n",
    "import matplotlib.pyplot as plt\n",
    "import copy\n",
    "from tqdm import tqdm\n",
    "\n",
    "# -----------------------------------------------------------------------------\n",
    "# CRITICAL: Add project root to path so we can import from backend\n",
    "# -----------------------------------------------------------------------------\n",
    "sys.path.append(os.path.abspath('..'))\n",
    "\n",
    "from backend.model import OsteoClassifier\n",
    "from backend.data_pipeline import apply_clahe\n",
    "from config.settings import IMG_SIZE, NORMALIZE_MEAN, NORMALIZE_STD, MODEL_PATH"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 1. Configuration & Hyperparameters\n",
    "Adjust `BATCH_SIZE` if you run out of memory (OOM). For a 4GB GPU, 16 is usually safe."
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "DEVICE = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n",
    "print(f\"Training on: {DEVICE}\")\n",
    "\n",
    "BATCH_SIZE = 16\n",
    "EPOCHS = 15\n",
    "LEARNING_RATE = 0.0001\n",
    "DATA_DIR = Path(\"../data/raw\")"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 2. Custom Data Transforms\n",
    "We integrate the CLAHE preprocessing logic here to ensure training data looks exactly like the data in the app."
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "class CLAHETransform:\n",
    "    \"\"\"Wrapper to apply CLAHE to PIL images in the transform pipeline\"\"\"\n",
    "    def __call__(self, img):\n",
    "        return apply_clahe(img)\n",
    "\n",
    "# Training Transforms (Augmentation + Preprocessing)\n",
    "train_transforms = transforms.Compose([\n",
    "    CLAHETransform(),\n",
    "    transforms.Resize(IMG_SIZE),\n",
    "    transforms.RandomHorizontalFlip(p=0.5),\n",
    "    transforms.RandomRotation(15),\n",
    "    transforms.ToTensor(),\n",
    "    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD)\n",
    "])\n",
    "\n",
    "# Validation Transforms (No Augmentation, Just Preprocessing)\n",
    "val_transforms = transforms.Compose([\n",
    "    CLAHETransform(),\n",
    "    transforms.Resize(IMG_SIZE),\n",
    "    transforms.ToTensor(),\n",
    "    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD)\n",
    "])"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 3. Data Loaders\n",
    "**Note:** Ensure your `data/raw` folder has `train` and `val` (or `test`) subfolders."
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transforms)\n",
    "val_dataset = datasets.ImageFolder(DATA_DIR / 'val', transform=val_transforms)\n",
    "\n",
    "train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)\n",
    "val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)\n",
    "\n",
    "print(f\"Training Samples: {len(train_dataset)}\")\n",
    "print(f\"Validation Samples: {len(val_dataset)}\")\n",
    "print(f\"Classes: {train_dataset.classes}\")"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 4. Model Initialization"
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "model = OsteoClassifier(num_classes=5)\n",
    "model = model.to(DEVICE)\n",
    "\n",
    "criterion = nn.CrossEntropyLoss()\n",
    "optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 5. Training Loop\n",
    "This loop saves the model to `models/densenet121_kl.pth` whenever validation accuracy improves."
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "best_acc = 0.0\n",
    "history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}\n",
    "\n",
    "print(\"Starting Training...\")\n",
    "\n",
    "for epoch in range(EPOCHS):\n",
    "    print(f\"Epoch {epoch+1}/{EPOCHS}\")\n",
    "    print(\"-\" * 10)\n",
    "\n",
    "    # --- TRAINING PHASE ---\n",
    "    model.train()\n",
    "    running_loss = 0.0\n",
    "    running_corrects = 0\n",
    "\n",
    "    for inputs, labels in tqdm(train_loader, desc=\"Training\"):\n",
    "        inputs = inputs.to(DEVICE)\n",
    "        labels = labels.to(DEVICE)\n",
    "\n",
    "        optimizer.zero_grad()\n",
    "\n",
    "        outputs = model(inputs)\n",
    "        loss = criterion(outputs, labels)\n",
    "        _, preds = torch.max(outputs, 1)\n",
    "\n",
    "        loss.backward()\n",
    "        optimizer.step()\n",
    "\n",
    "        running_loss += loss.item() * inputs.size(0)\n",
    "        running_corrects += torch.sum(preds == labels.data)\n",
    "\n",
    "    epoch_loss = running_loss / len(train_dataset)\n",
    "    epoch_acc = running_corrects.double() / len(train_dataset)\n",
    "\n",
    "    history['train_loss'].append(epoch_loss)\n",
    "    history['train_acc'].append(epoch_acc.item())\n",
    "\n",
    "    print(f\"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}\")\n",
    "\n",
    "    # --- VALIDATION PHASE ---\n",
    "    model.eval()\n",
    "    val_loss = 0.0\n",
    "    val_corrects = 0\n",
    "\n",
    "    with torch.no_grad():\n",
    "        for inputs, labels in tqdm(val_loader, desc=\"Validation\"):\n",
    "            inputs = inputs.to(DEVICE)\n",
    "            labels = labels.to(DEVICE)\n",
    "\n",
    "            outputs = model(inputs)\n",
    "            loss = criterion(outputs, labels)\n",
    "            _, preds = torch.max(outputs, 1)\n",
    "\n",
    "            val_loss += loss.item() * inputs.size(0)\n",
    "            val_corrects += torch.sum(preds == labels.data)\n",
    "\n",
    "    epoch_val_loss = val_loss / len(val_dataset)\n",
    "    epoch_val_acc = val_corrects.double() / len(val_dataset)\n",
    "\n",
    "    history['val_loss'].append(epoch_val_loss)\n",
    "    history['val_acc'].append(epoch_val_acc.item())\n",
    "\n",
    "    print(f\"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}\")\n",
    "\n",
    "    # --- SAVE BEST MODEL ---\n",
    "    if epoch_val_acc > best_acc:\n",
    "        best_acc = epoch_val_acc\n",
    "        # Ensure the directory exists\n",
    "        os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)\n",
    "        torch.save(model.state_dict(), MODEL_PATH)\n",
    "        print(f\"✅ New Best Model Saved to {MODEL_PATH}! (Acc: {best_acc:.4f})\")\n",
    "\n",
    "print(\"Training Complete.\")"
   ],
   "metadata": {}
  },
  {
   "cell_type": "markdown",
   "source": [
    "### 6. Performance Visualization"
   ],
   "metadata": {}
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "source": [
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.plot(history['train_loss'], label='Train Loss')\n",
    "plt.plot(history['val_loss'], label='Val Loss')\n",
    "plt.legend()\n",
    "plt.title('Loss Curves')\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "plt.plot(history['train_acc'], label='Train Acc')\n",
    "plt.plot(history['val_acc'], label='Val Acc')\n",
    "plt.legend()\n",
    "plt.title('Accuracy Curves')\n",
    "\n",
    "plt.show()"
   ],
   "metadata": {}
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}